This script is to collect data from Power BI exported files 
 
Data to be transformet and exported in XLS view, or can be copied directly from here
 

In [41]:
from pathlib import Path
import pandas as pd

Paths  
two files are exported from PowerBI,   
Structure list is created by You - use line schematics to fill all the structure names you need in the project 

In [52]:
in_dir = Path(r'C:\Users\Igor.Bertyaev.APD\OneDrive - APD\_IGOR\_python\PowerBI_parse\data_in')  # working directory
asset_xls = in_dir / 'Lines - Asset Attributes (Export to Excel).xlsx'
CA_xls = in_dir / 'Condition Assessment Overview.xlsx'
str_list_xls = in_dir / 'Structure_list.xlsx'
out_dir = Path(r'C:\Users\Igor.Bertyaev.APD\OneDrive - APD\_IGOR\_python\PowerBI_parse\data_out')  # output directory
final_xls = out_dir / 'PowerBI_Extract.xlsx'

# initial names
line_name = 'OTA-WKM-C'
circuit_1 = 'OHW-OTA-1'
circuit_2 = 'OHW-OTA-2'

Now we will use our structure list xls to create pandas table  
so, creating a new tab

In [43]:
# Read all sheets from str_list_xls to get column structures
str_sheets = pd.read_excel(str_list_xls, sheet_name=None)

# Create empty DataFrames for each tab with the same columns
str_dfs = {name: pd.DataFrame(columns=df.columns) for name, df in str_sheets.items()}

# Use the 'str_list' sheet as str_list_df
str_list_df = str_dfs.get('str_list', pd.DataFrame())

# Display the table
str_list_df

,structure_id,function,contract,type,att_type_phase,att_type_ew,BE,Leg_A,Leg_B,Leg_C,Leg_D,Strengthening


In [44]:
# Read str_list_df from Excel to get the structure list
str_list_df = pd.read_excel(str_list_xls, sheet_name='str_list')

print("str_list_df columns:", str_list_df.columns.tolist())

# Fill other columns from asset_xls Tower tab

# Mapping from Excel headers to DataFrame column names
column_mapping = {
    'Device Position': 'structure_id',
    'Tower Contract': 'contract',
    'Tower Type': 'type',
    'Insulator Attach Type': 'att_type_phase',
    'Earthwire Attach Type': 'att_type_ew',
    'Body Ext': 'BE',
    'Leg A Length': 'Leg_A',
    'Leg B Length': 'Leg_B',
    'Leg C Length': 'Leg_C',
    'Leg D Length': 'Leg_D',
    'Twr Strengthened Y/N': 'Strengthening'
    # Removed 'Circuit': 'circuit'
}

tower_df = pd.read_excel(asset_xls, sheet_name='Tower', usecols=list(column_mapping.keys()))
tower_df.rename(columns=column_mapping, inplace=True)

print("Tower df columns after rename:", tower_df.columns.tolist())

# Group by structure_id and aggregate
def aggregate_func(series):
    unique_vals = series.dropna().unique()
    if len(unique_vals) == 1:
        return unique_vals[0]
    else:
        return ' / '.join(map(str, unique_vals))

aggregated_df = tower_df.groupby('structure_id').agg(aggregate_func).reset_index()

print("Aggregated df columns:", aggregated_df.columns.tolist())

# Merge aggregated data with str_list_df
str_list_df = str_list_df.merge(aggregated_df, on='structure_id', how='left')

# Clean up overlapping columns by using the merged values
overlapping_cols = ['contract', 'type', 'att_type_phase', 'att_type_ew', 'BE', 'Leg_A', 'Leg_B', 'Leg_C', 'Leg_D', 'Strengthening']
for col in overlapping_cols:
    if col + '_y' in str_list_df.columns:
        str_list_df[col] = str_list_df[col + '_y']
        str_list_df.drop(columns=[col + '_x', col + '_y'], inplace=True, errors='ignore')

# Display updated table
str_list_df

str_list_df columns: ['structure_id', 'function', 'contract', 'type', 'att_type_phase', 'att_type_ew', 'BE', 'Leg_A', 'Leg_B', 'Leg_C', 'Leg_D', 'Strengthening']
Tower df columns after rename: ['structure_id', 'BE', 'att_type_ew', 'att_type_phase', 'Leg_A', 'Leg_B', 'Leg_C', 'Leg_D', 'contract', 'type', 'Strengthening']
Aggregated df columns: ['structure_id', 'BE', 'att_type_ew', 'att_type_phase', 'Leg_A', 'Leg_B', 'Leg_C', 'Leg_D', 'contract', 'type', 'Strengthening']


,structure_id,function,contract,type,att_type_phase,att_type_ew,BE,Leg_A,Leg_B,Leg_C,Leg_D,Strengthening
0,OTA-WKM-C0400,NaN,C385B,E,HSH,PLT,5.0,3.04,3.04,3.04,3.04,N
1,OTA-WKM-C0401,NaN,C385B,C,HBK,PLT,,6.08,6.08,6.08,6.08,N
2,OTA-WKM-C0402,NaN,C385B,B,HSH,PLT,,3.04,1.52,3.04,3.04,N
3,OTA-WKM-C0403,NaN,C385B,C,HBK,PLT,,3.04,3.04,3.04,3.04,N
4,OTA-WKM-C0404,NaN,C385B,A,HSH,PLT,,3.04,1.52,3.04,3.04,N
...,...,...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,NaN,C385B,A,PLT,PLT,3.0,6.08,6.08,6.08,6.08,N
88,OTA-WKM-C0488,NaN,C385B,E,PLT,PLT,,5.00,5.00,5.00,5.00,N
89,OTA-WKM-C0489,NaN,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,6.08,N
90,OTA-WKM-C0490,NaN,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,6.08,N


For future to update:
- Use not only Tower tab, but also Pole and Termination, as some structures may be placed there

Add columns from assets: 
- foundation - OK
- insulators - OK
- EW_assembly - ToDO
- Replace cirquit names with CCT1, CCT2
  
Create column 'function' based on insulators - OK


1. Add Foundation

In [45]:
# Add foundation column from 'Tower Foundation' tab
foundation_df = pd.read_excel(asset_xls, sheet_name='Tower Foundation', usecols=['Device Position', 'Foundation Type'])
foundation_df.rename(columns={'Device Position': 'structure_id', 'Foundation Type': 'foundation'}, inplace=True)

# Aggregate foundation if needed
def aggregate_func(series):
    unique_vals = series.dropna().unique()
    if len(unique_vals) == 1:
        return unique_vals[0]
    else:
        return ' / '.join(map(str, unique_vals))

foundation_agg = foundation_df.groupby('structure_id').agg({'foundation': aggregate_func}).reset_index()

# Replace existing foundation column with the new aggregated one
str_list_df.drop(columns=['foundation'], inplace=True, errors='ignore')
str_list_df = str_list_df.merge(foundation_agg[['structure_id', 'foundation']], on='structure_id', how='left')

# Display updated table
str_list_df

,structure_id,function,contract,type,att_type_phase,att_type_ew,BE,Leg_A,Leg_B,Leg_C,Leg_D,Strengthening,foundation
0,OTA-WKM-C0400,NaN,C385B,E,HSH,PLT,5.0,3.04,3.04,3.04,3.04,N,PBP
1,OTA-WKM-C0401,NaN,C385B,C,HBK,PLT,,6.08,6.08,6.08,6.08,N,PBP
2,OTA-WKM-C0402,NaN,C385B,B,HSH,PLT,,3.04,1.52,3.04,3.04,N,COG
3,OTA-WKM-C0403,NaN,C385B,C,HBK,PLT,,3.04,3.04,3.04,3.04,N,COG
4,OTA-WKM-C0404,NaN,C385B,A,HSH,PLT,,3.04,1.52,3.04,3.04,N,GRG
...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,NaN,C385B,A,PLT,PLT,3.0,6.08,6.08,6.08,6.08,N,COG
88,OTA-WKM-C0488,NaN,C385B,E,PLT,PLT,,5.00,5.00,5.00,5.00,N,GRG
89,OTA-WKM-C0489,NaN,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,6.08,N,GRG
90,OTA-WKM-C0490,NaN,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,6.08,N,PBP


2.1. Add ned DF with Insulators

In [ ]:
# Create insulators DataFrame from 'Insulators & Hardware' tab
ins_df = pd.read_excel(asset_xls, sheet_name='Insulators & Hardware', usecols=[
    'Circuit', 'Device Position', 'Jumper Ins Qty', 'Jumper Std Assy', 
    'Strain Back Ins Qty', 'Strain Back Std Assy', 'Strain Fwd Ins Qty', 
    'Strain Fwd Std Assy', 'Susp Ins Qty', 'Susp Std Assy', 'Weight Qty'
], dtype={
    'Jumper Ins Qty': 'Int64',
    'Strain Back Ins Qty': 'Int64',
    'Strain Fwd Ins Qty': 'Int64',
    'Susp Ins Qty': 'Int64'
})

ins_df.rename(columns={'Device Position': 'structure_id'}, inplace=True)

# Filter to only structures in str_list_df
ins_df = ins_df[ins_df['structure_id'].isin(str_list_df['structure_id'])]


# Convert columns to object dtype to avoid dtype warnings
for col in ins_df.columns:
    if col not in ['structure_id', 'Circuit']:
        ins_df[col] = ins_df[col].astype(object)

# Prefix circuit to each value in ins_df
for idx, row in ins_df.iterrows():
    circuit = row['Circuit']
    for col in ins_df.columns:
        if col not in ['structure_id', 'Circuit'] and pd.notna(row[col]):
            ins_df.at[idx, col] = f"{circuit}: {row[col]}"

# Drop Circuit column
ins_df.drop(columns=['Circuit'], inplace=True)

# Replace circuit names in ins_df with CCT1 and CCT2
ins_df = ins_df.apply(lambda col: col.str.replace(circuit_1, "CCT1", regex=False) if col.dtype == 'object' else col)
ins_df = ins_df.apply(lambda col: col.str.replace(circuit_2, "CCT2", regex=False) if col.dtype == 'object' else col)

# Aggregate to one row per structure_id
def combine_unique(series):
    vals = series.dropna().unique()
    return ' / '.join(map(str, vals)) if len(vals) > 0 else None

ins_agg = ins_df.groupby('structure_id').agg(combine_unique).reset_index()


# Display the aggregated insulators DataFrame
ins_agg

,structure_id,Jumper Ins Qty,Jumper Std Assy,Strain Back Ins Qty,Strain Back Std Assy,Strain Fwd Ins Qty,Strain Fwd Std Assy,Susp Ins Qty,Susp Std Assy,Weight Qty
0,OTA-WKM-C0400,CCT2: 42 / CCT1: 42,CCT2: 10A / CCT1: 848C,CCT2: 78 / CCT1: 84,CCT2: 13S / CCT1: 760C,CCT2: 78 / CCT1: 84,CCT2: 13S / CCT1: 760C,None,None,None
1,OTA-WKM-C0401,None,None,None,None,None,None,CCT1: 39 / CCT2: 39,CCT1: 11CM / CCT2: 11E,None
2,OTA-WKM-C0402,None,None,None,None,None,None,CCT2: 39 / CCT1: 39,CCT2: 10B / CCT1: 10B,None
3,OTA-WKM-C0403,None,None,None,None,None,None,CCT2: 39 / CCT1: 39,CCT2: 10B / CCT1: 10B,None
4,OTA-WKM-C0404,None,None,None,None,None,None,CCT1: 42 / CCT2: 42,CCT1: 11DM / CCT2: 11E,None
...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,None,None,None,None,None,None,CCT1: 3 / CCT2: 3,None,None
88,OTA-WKM-C0488,CCT2: 3 / CCT1: 3,CCT2: 523 / CCT1: 523,CCT2: 72 / CCT1: 72,CCT2: 13S / CCT1: 13S,CCT2: 72 / CCT1: 72,CCT2: 13S / CCT1: 13S,None,None,None
89,OTA-WKM-C0489,None,None,None,None,None,None,CCT2: 42 / CCT1: 42,CCT2: 10AS / CCT1: 10AS,None
90,OTA-WKM-C0490,None,None,None,None,None,None,CCT1: 3 / CCT2: 3,CCT1: 524 / CCT2: 524,None


2.2. Add Insulators to structure list df

In [47]:
# Add insulator columns to str_list_df
str_list_df = str_list_df.merge(ins_agg, on='structure_id', how='left')

# Fill 'function' column based on Strain columns
strain_cols = ['Strain Back Ins Qty', 'Strain Back Std Assy', 'Strain Fwd Ins Qty', 'Strain Fwd Std Assy']
str_list_df['function'] = str_list_df.apply(
    lambda row: "Strain" if any(pd.notna(row[col]) for col in strain_cols) else "Suspension",
    axis=1
)

# Display updated str_list_df
str_list_df

,structure_id,function,contract,type,att_type_phase,att_type_ew,BE,Leg_A,Leg_B,Leg_C,...,foundation,Jumper Ins Qty,Jumper Std Assy,Strain Back Ins Qty,Strain Back Std Assy,Strain Fwd Ins Qty,Strain Fwd Std Assy,Susp Ins Qty,Susp Std Assy,Weight Qty
0,OTA-WKM-C0400,Strain,C385B,E,HSH,PLT,5.0,3.04,3.04,3.04,...,PBP,OHW-OTA-2: 42 / OHW-OTA-1: 42,OHW-OTA-2: 10A / OHW-OTA-1: 848C,OHW-OTA-2: 78 / OHW-OTA-1: 84,OHW-OTA-2: 13S / OHW-OTA-1: 760C,OHW-OTA-2: 78 / OHW-OTA-1: 84,OHW-OTA-2: 13S / OHW-OTA-1: 760C,None,None,None
1,OTA-WKM-C0401,Suspension,C385B,C,HBK,PLT,,6.08,6.08,6.08,...,PBP,None,None,None,None,None,None,OHW-OTA-1: 39 / OHW-OTA-2: 39,OHW-OTA-1: 11CM / OHW-OTA-2: 11E,None
2,OTA-WKM-C0402,Suspension,C385B,B,HSH,PLT,,3.04,1.52,3.04,...,COG,None,None,None,None,None,None,OHW-OTA-2: 39 / OHW-OTA-1: 39,OHW-OTA-2: 10B / OHW-OTA-1: 10B,None
3,OTA-WKM-C0403,Suspension,C385B,C,HBK,PLT,,3.04,3.04,3.04,...,COG,None,None,None,None,None,None,OHW-OTA-2: 39 / OHW-OTA-1: 39,OHW-OTA-2: 10B / OHW-OTA-1: 10B,None
4,OTA-WKM-C0404,Suspension,C385B,A,HSH,PLT,,3.04,1.52,3.04,...,GRG,None,None,None,None,None,None,OHW-OTA-1: 42 / OHW-OTA-2: 42,OHW-OTA-1: 11DM / OHW-OTA-2: 11E,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,OTA-WKM-C0487,Suspension,C385B,A,PLT,PLT,3.0,6.08,6.08,6.08,...,COG,None,None,None,None,None,None,OHW-OTA-1: 3 / OHW-OTA-2: 3,None,None
88,OTA-WKM-C0488,Strain,C385B,E,PLT,PLT,,5.00,5.00,5.00,...,GRG,OHW-OTA-2: 3 / OHW-OTA-1: 3,OHW-OTA-2: 523 / OHW-OTA-1: 523,OHW-OTA-2: 72 / OHW-OTA-1: 72,OHW-OTA-2: 13S / OHW-OTA-1: 13S,OHW-OTA-2: 72 / OHW-OTA-1: 72,OHW-OTA-2: 13S / OHW-OTA-1: 13S,None,None,None
89,OTA-WKM-C0489,Suspension,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,...,GRG,None,None,None,None,None,None,OHW-OTA-2: 42 / OHW-OTA-1: 42,OHW-OTA-2: 10AS / OHW-OTA-1: 10AS,None
90,OTA-WKM-C0490,Suspension,C385B,A,HSH,PLT,5.0,6.08,6.08,6.08,...,PBP,None,None,None,None,None,None,OHW-OTA-1: 3 / OHW-OTA-2: 3,OHW-OTA-1: 524 / OHW-OTA-2: 524,None


3. Add EW

4. Write to Excel

In [ ]:
# Export str_list_df back to str_list_xls 'str_list' sheet
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# Load the workbook
wb = load_workbook(str_list_xls)

# Remove the existing 'str_list' sheet if it exists
if 'str_list_filled' in wb.sheetnames:
    wb.remove(wb['str_list_filled'])

# Create new 'str_list' sheet
ws = wb.create_sheet('str_list_filled')

# Write str_list_df to the sheet
for r in dataframe_to_rows(str_list_df, index=False, header=True):
    ws.append(r)

# Save the workbook
wb.save(str_list_xls)

print("Data exported to str_list_xls 'str_list_filled' sheet.")

Data exported to str_list_xls 'str_list' sheet.
